In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('D:/data_project/raw-data/kaggle_transactions.csv') 
df = df.rename(columns={'cc_num': 'customer_id'})

In [ ]:
customers = df[['customer_id', 'first', 'last', 'gender', 'dob', 'job',
                 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop']]
customers = customers.drop_duplicates(subset='customer_id')
customers = customers.rename(columns={'first': 'first_name', 'last': 'last_name'})
print(customers.shape)

In [ ]:
df['merchant_clean'] = df['merchant'].str.replace('fraud_', '', regex=False)

merchants = df.groupby('merchant_clean').agg(
    merch_lat=('merch_lat', 'mean'),
    merch_long=('merch_long', 'mean'),
    merch_zipcode=('merch_zipcode', 'first')
).reset_index()

In [ ]:
merchants['merchant_id'] = pd.factorize(merchants['merchant_clean'])[0] + 1
merchants = merchants[['merchant_id', 'merchant_clean', 'merch_lat', 'merch_long', 'merch_zipcode']]
print(merchants.shape)

In [ ]:
df = df.merge(merchants[['merchant_clean', 'merchant_id']], on='merchant_clean', how='left')
print(df['merchant_id'].isnull().sum())

In [ ]:
df['txn_date'] = pd.to_datetime(df['trans_date_trans_time'])

In [ ]:
transactions = df[['trans_num', 'customer_id', 'merchant_id', 'txn_date', 'amt', 'category', 'is_fraud']]

In [ ]:
transactions = transactions.rename(columns={'trans_num': 'trans_id', 'amt': 'amount'})

In [ ]:
print(transactions.shape)  # expect (1296675, 7)
print(transactions['customer_id'].isin(customers['customer_id']).all())  
print(transactions['merchant_id'].isin(merchants['merchant_id']).all())

In [ ]:
sample_customers = customers['customer_id'].sample(200, random_state=42)

In [ ]:
transactions_slice = transactions[transactions['customer_id'].isin(sample_customers)]

In [ ]:
customers_slice = customers[customers['customer_id'].isin(sample_customers)]

In [ ]:
used_merchants = transactions_slice['merchant_id'].unique()
merchants_slice = merchants[merchants['merchant_id'].isin(used_merchants)]

In [ ]:
print(transactions_slice.shape)   
print(customers_slice.shape)      
print(merchants_slice.shape)

In [ ]:
print(transactions_slice['customer_id'].isin(customers_slice['customer_id']).all())  # expect True
print(transactions_slice['merchant_id'].isin(merchants_slice['merchant_id']).all())  # expect True


In [ ]:
customers_slice.to_csv('D:/sql-finance-analysis/data/customers.csv', index=False)
merchants_slice.to_csv('D:/sql-finance-analysis/data/merchants.csv', index=False)
transactions_slice.to_csv('D:/sql-finance-analysis/data/transactions.csv', index=False)